# Beyond Genre

Use Spotify audio features to cluster tracks and estimate popularity. Run the cells in order from this folder with the CSV in `data/`. The first clustering analysis describes the entire dataset. The final cell refits scaling and clustering on the training split for evaluation.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load the dataset
df = pd.read_csv('data/dataset.csv', low_memory=False)

# 2. Data cleaning
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
df = df.drop_duplicates(subset=['track_id'])
df = df.dropna()

# 3. Feature Selection
cols_to_drop = ['track_id', 'artists', 'album_name', 'track_name']
df_ml = df.drop(columns=cols_to_drop).copy()

# 4. Encode categorical variables
df_ml['explicit'] = df_ml['explicit'].astype(int)
le = LabelEncoder()
df_ml['track_genre'] = le.fit_transform(df_ml['track_genre'])

print("Data Cleaning and Processing Complete!")
print(f"{len(df_ml):,} unique, complete tracks across {df.track_genre.nunique()} retained genres.")

In [ ]:
# Chart 1: Distribution of Popularity (The target variable)
plt.figure(figsize=(10, 5))
sns.histplot(df_ml['popularity'], bins=30, kde=True, color='teal')
plt.title('Distribution of Song Popularity')
plt.show()

# Chart 2: Popularity by Top 10 Genres
top_genres = df['track_genre'].value_counts().nlargest(10).index
df_top_genres = df[df['track_genre'].isin(top_genres)]

plt.figure(figsize=(12, 6))
sns.boxplot(x='track_genre', y='popularity', data=df_top_genres, hue='track_genre', legend=False, palette='viridis')
plt.xticks(rotation=45)
plt.title('Popularity Spread Across Top 10 Genres')
plt.show()

# Chart 3: Correlation Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(df_ml.corr(), annot=False, cmap='coolwarm')
plt.title("Correlation Heatmap of Audio Features")
plt.show()

In [ ]:
# 1. Feature Scaling
features = df_ml.drop(columns=['popularity'])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# 2. Elbow Method
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(scaled_features)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), wcss, marker='o')
plt.title('Elbow Method for Optimal Clusters')
plt.show()

# 3. Applying K-Means (K=5)
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_ml['vibe_cluster'] = kmeans.fit_predict(scaled_features)

# Chart 4: Visualizing Cluster Differences (Proving the 'Vibe')
cluster_means = df_ml.groupby('vibe_cluster').mean().drop(columns=['popularity'])
cluster_means[['energy', 'danceability', 'acousticness', 'instrumentalness']].plot(kind='bar', figsize=(12, 6))
plt.title('Audio Feature Characteristics per Vibe Cluster')
plt.ylabel('Mean Score')
plt.show()

In [ ]:
# Refit the cluster feature on training rows for this held-out evaluation.
# The integer genre codes follow the original course notebook.
X_train, X_test, y_train, y_test = train_test_split(
    features, df_ml["popularity"], test_size=0.2, random_state=42
)
train_scaler = StandardScaler().fit(X_train)
train_clusters = KMeans(n_clusters=5, random_state=42, n_init=10)
X_train = X_train.assign(vibe_cluster=train_clusters.fit_predict(train_scaler.transform(X_train)))
X_test = X_test.assign(vibe_cluster=train_clusters.predict(train_scaler.transform(X_test)))
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 3. Evaluate
y_pred = lr_model.predict(X_test)
print(f"Held-out R²: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} popularity points")
baseline = np.full(len(y_test), y_train.mean())
print(f"Training-mean baseline RMSE: {np.sqrt(mean_squared_error(y_test, baseline)):.2f}")

# Chart 5: Predicted vs Actual Popularity (Final Validation)
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2)
plt.title('Actual vs. Predicted Popularity')
plt.xlabel('Actual Popularity')
plt.ylabel('Predicted Popularity')
plt.show()